# Clustering Urban Mobility Patterns Using Bike-Sharing Data
**Course assignment notebook — built step by step**

This notebook follows the assignment structure exactly:
1. Data Preparation & Feature Engineering
2. Clustering Methods (K-Means, AGNES, DBSCAN)
3. Evaluation & Comparison
4. Cluster Interpretation & Validation
5. Theoretical Questions (answered in the report, referenced here)


## 0. Setup
Import libraries and set a fixed random seed for reproducibility.

In [1]:
# Libraries will be added here as we need them, one step at a time.
import numpy as np
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


## Part 1: Data Preparation and Feature Engineering

### 1.1 Inspect and Clean the Data
- Load `hour.csv`
- Check missing values, invalid values, duplicates
- Convert `dteday` to datetime
- Verify each date has a reasonable number of hourly observations
- Identify and handle unusual/incomplete days

In [8]:
df = pd.read_csv("data/hour.csv") 

df['dteday'] = pd.to_datetime(df['dteday'])
print("Shape of the dataset:", df.shape)

# Check for missing values
total_missing = df.isnull().sum().sum()
print(f"\nTotal missing values in the dataset: {total_missing}")

# Check for duplicate rows, and duplicate (date, hour) pairs
duplicated_rows = df.duplicated().sum()
print(f"\nNumber of duplicated rows in the dataset: {duplicated_rows}")

duplicate_date_hour = df.duplicated(subset=['dteday', 'hr']).sum()
print(f"Duplicate (date, hour) pairs: {duplicate_date_hour}\n")

for col in ['temp', 'atemp', 'hum', 'windspeed']:
    print(f"{col}: min={df[col].min()}, max={df[col].max()}")

hourly_counts = df.groupby('dteday').size()
print(f"\nHourly counts: {hourly_counts.shape[0]}")
print(f"Days with all 24 hours present: {(hourly_counts == 24).sum()}")
print(f"Days missing at least 1 hour: {(hourly_counts < 24).sum()}")

# Decide a cutoff for "usable" days
MIN_HOURS_PER_DAY = 22
complete_days = hourly_counts[hourly_counts >= MIN_HOURS_PER_DAY].index
incomplete_days = hourly_counts[hourly_counts < MIN_HOURS_PER_DAY]

print(f"\nDays kept (>= {MIN_HOURS_PER_DAY} hours): {len(complete_days)}")
print(f"Days dropped (< {MIN_HOURS_PER_DAY} hours): {len(incomplete_days)}")
print(incomplete_days.sort_values())

# clean data
df_clean = df[df['dteday'].isin(complete_days)].copy()
print("\nRows remaining after filtering:", df_clean.shape[0])

Shape of the dataset: (17379, 17)

Total missing values in the dataset: 0

Number of duplicated rows in the dataset: 0
Duplicate (date, hour) pairs: 0

temp: min=0.02, max=1.0
atemp: min=0.0, max=1.0
hum: min=0.0, max=1.0
windspeed: min=0.0, max=0.8507

Hourly counts: 731
Days with all 24 hours present: 655
Days missing at least 1 hour: 76

Days kept (>= 22 hours): 723
Days dropped (< 22 hours): 8
dteday
2012-10-29     1
2011-01-27     8
2012-10-30    11
2011-01-18    12
2011-01-26    16
2011-08-28    17
2011-02-22    18
2011-08-27    18
dtype: int64

Rows remaining after filtering: 17278


### 1.2 Generate Daily Mobility Features
Aggregate hourly records into one row per day.

In [10]:
# Aggregate the cleaned hourly records into one row per day 
daily_agg = df_clean.groupby('dteday').agg(
    total_rentals=('cnt', 'sum'),
    avg_hourly_rentals=('cnt', 'mean'),
    max_hourly_rentals=('cnt', 'max'),
    std_hourly_rentals=('cnt', 'std'),
    total_casual=('casual', 'sum'),
    total_registered=('registered', 'sum'),
    avg_temp=('temp', 'mean'),
    avg_atemp=('atemp', 'mean'),
    avg_hum=('hum', 'mean'),
    avg_windspeed=('windspeed', 'mean'),
    season=('season', 'first'),
    yr=('yr', 'first'),
    mnth=('mnth', 'first'),
    holiday=('holiday', 'first'),
    weekday=('weekday', 'first'),
    workingday=('workingday', 'first'),
    weathersit=('weathersit', lambda x: x.mode()[0])  
).reset_index()

# Filter Morning and evening peak hours 
morning_sum = df_clean[df_clean['hr'].between(6, 9)].groupby('dteday')['cnt'].sum().rename('morning_rentals')
evening_sum = df_clean[df_clean['hr'].between(16, 19)].groupby('dteday')['cnt'].sum().rename('evening_rentals')

# Merge to daily_agg
daily_agg = daily_agg.merge(morning_sum, on='dteday', how='left').merge(evening_sum, on='dteday', how='left')
daily_agg[['morning_rentals', 'evening_rentals']] = daily_agg[['morning_rentals', 'evening_rentals']].fillna(0)

daily_agg['morning_peak_share'] = daily_agg['morning_rentals'] / daily_agg['total_rentals']
daily_agg['evening_peak_share'] = daily_agg['evening_rentals'] / daily_agg['total_rentals']
daily_agg['offpeak_share'] = 1 - daily_agg['morning_peak_share'] - daily_agg['evening_peak_share']

# Casual and registered user proportions 
daily_agg['casual_prop'] = daily_agg['total_casual'] / daily_agg['total_rentals']
daily_agg['registered_prop'] = daily_agg['total_registered'] / daily_agg['total_rentals']

print("\nDaily aggregated dataset shape:", daily_agg.shape)
daily_agg.head()


Daily aggregated dataset shape: (723, 25)


,dteday,total_rentals,avg_hourly_rentals,max_hourly_rentals,std_hourly_rentals,total_casual,total_registered,avg_temp,avg_atemp,avg_hum,...,weekday,workingday,weathersit,morning_rentals,evening_rentals,morning_peak_share,evening_peak_share,offpeak_share,casual_prop,registered_prop
0,2011-01-01,985,41.041667,110,34.292196,331,654,0.344167,0.363625,0.805833,...,6,0,1,27,232,0.027411,0.235533,0.737056,0.336041,0.663959
1,2011-01-02,801,34.826087,93,29.785067,131,670,0.363478,0.353739,0.696087,...,0,0,2,31,224,0.038702,0.279650,0.681648,0.163546,0.836454
2,2011-01-03,1349,61.318182,157,48.792453,120,1229,0.196364,0.189405,0.437273,...,1,1,1,336,500,0.249073,0.370645,0.380282,0.088955,0.911045
3,2011-01-04,1562,67.913043,212,59.889985,108,1454,0.200000,0.212122,0.590435,...,2,1,1,409,589,0.261844,0.377081,0.361076,0.069142,0.930858
4,2011-01-05,1600,69.565217,195,58.427753,82,1518,0.226957,0.229270,0.436957,...,3,1,1,431,580,0.269375,0.362500,0.368125,0.051250,0.948750


### 1.3 Prepare the Clustering Data
- Check distributions / skew
- Handle correlated or redundant features
- Standardize features
- Justify any removed features

In [ ]:
# Step 3 code goes here.


### 1.4 Exploratory Visualisation
- Histograms / boxplots
- Correlation heatmap
- Daily rental pattern plots
- PCA 2D visualisation

In [ ]:
# Step 4 code goes here.


## Part 2: Apply and Analyse Clustering Methods

### 2.1 K-Means Clustering

In [ ]:
# Step 5 code goes here.


### 2.2 Hierarchical Clustering (AGNES)

In [ ]:
# Step 6 code goes here.


### 2.3 DBSCAN

In [ ]:
# Step 7 code goes here.


## Part 3: Clustering Evaluation and Comparison

In [ ]:
# Step 8 code goes here.


## Part 4: Cluster Interpretation and Validation

In [ ]:
# Step 9 code goes here.


## Part 5: Theoretical Understanding
Written answers to the 10 theory questions — drafted together with the report in Step 10.